In [2]:
from utils import get_dataset_lines

# Alignment with Affine Gap Penalties Problem
**Code Challenge**: Solve the Alignment with Affine Gap Penalties Problem.

**Input**: A match reward, a mismatch penalty, a gap opening penalty, a gap extension penalty, and two nucleotide strings.

**Output**: The maximum alignment score between *v* and *w*, followed by an alignment of *v* and *w* achieving this maximum score.

**Sample Input**:

```
1 3 2 1
GA
GTTA
```

**Sample Output**:

```
-1
G--A
GTTA
```

In [3]:
def GlobalAlignmentAffineGap(match_reward, mismatch_penalty, gap_open, gap_extension, v, w):
    n = len(v)
    m = len(w)
    
    neg_inf = -float('inf')
    
    # Matrices
    # lower: vertical (gap in w)
    # upper: horizontal (gap in v)
    # middle: diagonal (match/mismatch)
    lower = [[neg_inf] * (m + 1) for _ in range(n + 1)]
    middle = [[neg_inf] * (m + 1) for _ in range(n + 1)]
    upper = [[neg_inf] * (m + 1) for _ in range(n + 1)]
    
    # Backtrack
    # lower_bt: 0=extend, 1=open (from middle)
    # upper_bt: 0=extend, 1=open (from middle)
    # middle_bt: 0=from lower, 1=from middle, 2=from upper
    lower_bt = [[0] * (m + 1) for _ in range(n + 1)]
    upper_bt = [[0] * (m + 1) for _ in range(n + 1)]
    middle_bt = [[0] * (m + 1) for _ in range(n + 1)]
    
    # Initialization
    middle[0][0] = 0
    
    # Initialize first column (vertical, gap in w)
    for i in range(1, n + 1):
        lower[i][0] = -gap_open - (i - 1) * gap_extension
        middle[i][0] = lower[i][0]
        upper[i][0] = neg_inf 
        
        lower_bt[i][0] = 1 if i == 1 else 0
        middle_bt[i][0] = 0 # from lower
        
    # Initialize first row (horizontal, gap in v)
    for j in range(1, m + 1):
        upper[0][j] = -gap_open - (j - 1) * gap_extension
        middle[0][j] = upper[0][j]
        lower[0][j] = neg_inf
        
        upper_bt[0][j] = 1 if j == 1 else 0
        middle_bt[0][j] = 2 # from upper
        
    # Fill matrices
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Lower
            extend = lower[i-1][j] - gap_extension
            open_gap = middle[i-1][j] - gap_open
            if extend >= open_gap:
                lower[i][j] = extend
                lower_bt[i][j] = 0
            else:
                lower[i][j] = open_gap
                lower_bt[i][j] = 1
                
            # Upper
            extend = upper[i][j-1] - gap_extension
            open_gap = middle[i][j-1] - gap_open
            if extend >= open_gap:
                upper[i][j] = extend
                upper_bt[i][j] = 0
            else:
                upper[i][j] = open_gap
                upper_bt[i][j] = 1
                
            # Middle
            score = match_reward if v[i-1] == w[j-1] else -mismatch_penalty
            
            val_l = lower[i-1][j-1] + score
            val_m = middle[i-1][j-1] + score
            val_u = upper[i-1][j-1] + score
            
            best = val_m
            state = 1
            
            if val_l > best:
                best = val_l
                state = 0
            if val_u > best:
                best = val_u
                state = 2
                
            middle[i][j] = best
            middle_bt[i][j] = state
            
    # Backtrack
    score_l = lower[n][m]
    score_m = middle[n][m]
    score_u = upper[n][m]
    
    max_score = score_m
    state = 1
    
    if score_l > max_score:
        max_score = score_l
        state = 0
    if score_u > max_score:
        max_score = score_u
        state = 2
        
    aligned_v = []
    aligned_w = []
    
    i, j = n, m
    
    while i > 0 or j > 0:
        if state == 0: # Lower (gap in w, vertical)
            aligned_v.append(v[i-1])
            aligned_w.append('-')
            if lower_bt[i][j] == 1: # open
                state = 1 # to middle
            else:
                state = 0 # stay lower
            i -= 1
        elif state == 2: # Upper (gap in v, horizontal)
            aligned_v.append('-')
            aligned_w.append(w[j-1])
            if upper_bt[i][j] == 1: # open
                state = 1 # to middle
            else:
                state = 2 # stay upper
            j -= 1
        else: # Middle
            aligned_v.append(v[i-1])
            aligned_w.append(w[j-1])
            state = middle_bt[i][j]
            i -= 1
            j -= 1
            
    return max_score, "".join(reversed(aligned_v)), "".join(reversed(aligned_w))

In [34]:
# Sample Input
match_reward = 1
mismatch_penalty = 3
gap_open = 2
gap_extension = 1
v = "GA"
w = "GTTA"

# Run the function
score, align_v, align_w = GlobalAlignmentAffineGap(match_reward, mismatch_penalty, gap_open, gap_extension, v, w)
print(score)
print(align_v)
print(align_w)

# Test Assertion
expected_score = -1
# Note: Alignment might vary, but score should be exact
assert score == expected_score, f"Expected score {expected_score}, but got {score}"
print("Test passed!")

-1
G--A
GTTA
Test passed!


In [5]:
# Test Dataset
# ERROR: We can not prepare challenge for you right now. Probably, server overloaded. Please try again later.
test_dataset_filename = 'dataset_30198_8.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    match_reward, mismatch_penalty, gap_open, gap_extension = map(int, lines[0].split())
    v = lines[1].strip()
    w = lines[2].strip()
    
    score, align_v, align_w = GlobalAlignmentAffineGap(match_reward, mismatch_penalty, gap_open, gap_extension, v, w)
    print(score)
    print(align_v)
    print(align_w)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

File dataset_30198_8.txt not found. Please download the dataset and check the filename.


# Middle Edge in Linear Space Problem
**Code Challenge**: Solve the Middle Edge in Linear Space Problem (for protein strings).

**Input**: A match reward, a mismatch penalty, an indel penalty, and two nucleotide strings.

**Output**: A middle edge in the alignment graph.

**Sample Input**:

```
1 1 2
GAGA
GAT
```

**Sample Output**:

```
2 2
2 3
```

In [24]:
def GetScoreColumn(match_reward, mismatch_penalty, indel_penalty, v, w):
    n = len(v)
    m = len(w)
    
    current_col = [0] * (n + 1)
    
    # Initialize first column (j=0)
    for i in range(1, n + 1):
        current_col[i] = current_col[i-1] - indel_penalty
        
    for j in range(1, m + 1):
        next_col = [0] * (n + 1)
        # Initialize first row element (i=0)
        next_col[0] = current_col[0] - indel_penalty
        
        for i in range(1, n + 1):
            match = current_col[i-1] + (match_reward if v[i-1] == w[j-1] else -mismatch_penalty)
            delete = next_col[i-1] - indel_penalty # Vertical (gap in w)
            insert = current_col[i] - indel_penalty # Horizontal (gap in v)
            next_col[i] = max(match, delete, insert)
        current_col = next_col
        
    return current_col

def MiddleEdge(match_reward, mismatch_penalty, indel_penalty, v, w):
    # Ensure v is the shorter string to optimize space
    # The sample output suggests that the coordinates should be returned 
    # with respect to the orientation where v is the shorter string.
    if len(v) > len(w):
        v, w = w, v
        
    m = len(w)
    middle = (m + 1) // 2
    
    # Left scores at column middle
    scores_left = GetScoreColumn(match_reward, mismatch_penalty, indel_penalty, v, w[:middle])
    
    # Right scores at column middle + 1 (from sink)
    scores_right = GetScoreColumn(match_reward, mismatch_penalty, indel_penalty, v[::-1], w[middle:][::-1])
    
    n = len(v)
    max_score = -float('inf')
    best_i = -1
    edge_type = -1 # 0: Horizontal, 1: Diagonal
    
    for i in range(n + 1):
        # Horizontal edge: (i, middle) -> (i, middle+1)
        # Score = Left[i] - indel + Right[n-i]
        score_h = scores_left[i] - indel_penalty + scores_right[n-i]
        if score_h > max_score:
            max_score = score_h
            best_i = i
            edge_type = 0
            
        # Diagonal edge: (i, middle) -> (i+1, middle+1)
        if i < n:
            score_d = scores_left[i] + (match_reward if v[i] == w[middle] else -mismatch_penalty) + scores_right[n-(i+1)]
            if score_d > max_score:
                max_score = score_d
                best_i = i
                edge_type = 1
                
    if edge_type == 0:
        return (best_i, middle), (best_i, middle + 1)
    else:
        return (best_i, middle), (best_i + 1, middle + 1)

In [29]:
# Sample Input
match_reward = 1
mismatch_penalty = 1
indel_penalty = 2
v = "GAGA"
w = "GAT"

# Run the function
# MiddleEdge swaps internally if len(v) > len(w)
start, end = MiddleEdge(match_reward, mismatch_penalty, indel_penalty, v, w)

print(f"{start[0]} {start[1]}")
print(f"{end[0]} {end[1]}")

# Test Assertion
expected_start = (2, 2)
expected_end = (2, 3)

assert start == expected_start, f"Expected start {expected_start}, but got {start}"
assert end == expected_end, f"Expected end {expected_end}, but got {end}"
print("Test passed!")

2 2
2 3
Test passed!


In [26]:
# Test Dataset
test_dataset_filename = 'dataset_30202_12.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    match_reward, mismatch_penalty, indel_penalty = map(int, lines[0].split())
    v = lines[1].strip()
    w = lines[2].strip()
    
    start, end = MiddleEdge(match_reward, mismatch_penalty, indel_penalty, v, w)
    print(f"{start[0]} {start[1]}")
    print(f"{end[0]} {end[1]}")
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

438 450
439 451


# Linear Space Alignment
**Code Challenge**: Implement LinearSpaceAlignment to solve the Global Alignment Problem for a large dataset.

**Input**: A match reward, a mismatch penalty, an indel penalty, and two long (2000 nucleotide) DNA strings.

**Output**: The maximum alignment score of these strings, followed by an alignment achieving this maximum score.

**Sample Input**:

```
1 1 2
GAGA
GAT
```

**Sample Output**:

```
-1
GAGA
GA-T
```

In [48]:
def MiddleEdge(v, w, top, bottom, left, right, match_reward, mismatch_penalty, indel_penalty):
    v_sub = v[top:bottom]
    
    middle = (left + right) // 2
    
    # Scores from source to column middle
    scores_left = GetScoreColumn(match_reward, mismatch_penalty, indel_penalty, v_sub, w[left:middle])
    
    # Scores from sink to column middle + 1
    scores_right = GetScoreColumn(match_reward, mismatch_penalty, indel_penalty, v_sub[::-1], w[middle+1:right][::-1])
    
    n = len(v_sub)
    max_score = -float('inf')
    best_i = -1
    edge_type = '' 
    
    for i in range(n + 1):
        # Horizontal edge: (i, middle) -> (i, middle+1)
        score_h = scores_left[i] - indel_penalty + scores_right[n-i]
        if score_h > max_score:
            max_score = score_h
            best_i = i
            edge_type = 'H'
            
        # Diagonal edge: (i, middle) -> (i+1, middle+1)
        if i < n:
            score_d = scores_left[i] + (match_reward if v_sub[i] == w[middle] else -mismatch_penalty) + scores_right[n-(i+1)]
            if score_d > max_score:
                max_score = score_d
                best_i = i
                edge_type = 'D'
                
    return edge_type, top + best_i

def LinearSpaceAlignment(v, w, top, bottom, left, right, match_reward, mismatch_penalty, indel_penalty):
    if left == right:
        return "V" * (bottom - top)
    if top == bottom:
        return "H" * (right - left)
        
    middle = (left + right) // 2
    edge_type, midNode = MiddleEdge(v, w, top, bottom, left, right, match_reward, mismatch_penalty, indel_penalty)
    
    path1 = LinearSpaceAlignment(v, w, top, midNode, left, middle, match_reward, mismatch_penalty, indel_penalty)
    
    next_node = midNode + (1 if edge_type == 'D' else 0)
    
    path2 = LinearSpaceAlignment(v, w, next_node, bottom, middle + 1, right, match_reward, mismatch_penalty, indel_penalty)
    
    return path1 + edge_type + path2

def PathToAlignment(v, w, path, match_reward, mismatch_penalty, indel_penalty):
    align_v = ""
    align_w = ""
    score = 0
    i = 0
    j = 0
    
    for move in path:
        if move == 'D':
            align_v += v[i]
            align_w += w[j]
            score += match_reward if v[i] == w[j] else -mismatch_penalty
            i += 1
            j += 1
        elif move == 'V':
            align_v += v[i]
            align_w += '-'
            score -= indel_penalty
            i += 1
        elif move == 'H':
            align_v += '-'
            align_w += w[j]
            score -= indel_penalty
            j += 1
            
    return score, align_v, align_w

In [50]:
# Sample Input
match_reward = 1
mismatch_penalty = 1
indel_penalty = 2
v = "GAGA"
w = "GAT"

# Run the function
path = LinearSpaceAlignment(v, w, 0, len(v), 0, len(w), match_reward, mismatch_penalty, indel_penalty)
score, align_v, align_w = PathToAlignment(v, w, path, match_reward, mismatch_penalty, indel_penalty)

print(score)
print(align_v)
print(align_w)

# Test Assertion
expected_score = -1

assert score == expected_score, f"Expected score {expected_score}, but got {score}"

if align_w == "GA-T":
    print("Matched sample output exactly.")
elif align_w == "GAT-":
    print("Matched alternative optimal alignment.")
else:
    raise AssertionError(f"Unexpected alignment: {align_w}")
    
print("Test passed!")

-1
GAGA
GAT-
Matched alternative optimal alignment.
Test passed!


In [52]:
# Test Dataset
test_dataset_filename = 'dataset_30202_14.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    match_reward, mismatch_penalty, indel_penalty = map(int, lines[0].split())
    v = lines[1].strip()
    w = lines[2].strip()
    
    path = LinearSpaceAlignment(v, w, 0, len(v), 0, len(w), match_reward, mismatch_penalty, indel_penalty)
    score, align_v, align_w = PathToAlignment(v, w, path, match_reward, mismatch_penalty, indel_penalty)
    
    print(score)
    print(align_v)
    print(align_w)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

-370
ATGGAATT-G----GGTGGCCACAAA-GAG-A-AACGGTGT-C-CAG-TATTCACATCCCTTTTGTGGCTCAAATTAACAATTGCTTGACACATGTACGATTTTATAAACGGTCGAAGGT-GTA-ATCGC--A-ATTTGAGCATGGTGACCACA---GACCTGTTTGCCCGACACCGTCCA--------TT-GATTCAC-G-C-----AC--T---TTAGCTG-TCCTACAGAGC-----CCTCTCACTTCA-G-GC--TACCGACATACTTCCGAACGCTCCCCGGTCTTCAACCGCTTCTCG-GCATCCGATAGG--C-GGACACCCACTCTAATGCCGTTAGCCCGTGTGGTTGGCAAGTTGCAAGGCTTATAATGGCACCCGCCCAAAAGAAAACTCTTT---GG-TTACGCGCACTCAAAC-A-CAA-GATCTTTGGCATTTA-GC-TCACACA-ACTCGGTAGACATGGATGCATTCAGGATGCTTAAACCTACAACATTTTAATC-GCGTGAGATATGTCTACCCTTTCGCGTGACAAGTCCTCGACTGT-CTACGTAAACGGGCCCGATAAGGTGTTAGTGGTCCAGCGTCGGTAGTGCTACATCGGT-G-GTCGTCAGGG-CCCCG-GCTGTTTACTT-GGGG--CGTCCAG-G-GGCCTAGCCAGACGATGAACTTGTTGCATATTTCAGTCTTTGAATCAGGACAGATACAACACGTAGTTTATGGTAGTAGCCACTACTTATGATAGTTCGGCGCGTATACTCAATGGTATGACCCGGATCCGA-CTCCGAATCGATTACTTAATCAGTCTACGAGGCAAGACTAAGACCTGAATTGAGAACGACGATCACCTTAGGATGCGCCGCCGAACCGAACGA-GTGCTTGGTGTAGGAGCTGTTGGTGAACCCTCTCGACACCGGGACTTCCTAGCG-TA-GGCCAAACTAGTGACAT-TTC-TATG-GTCGCAGGCGTCCCAG-TC

# Multiple Longest Common Subsequence Problem
In the Multiple Longest Common Subsequence Problem, the score of a column of the alignment matrix is equal to 1 if all of the column's symbols are identical, and 0 if even one symbol disagrees.

**Code Challenge**: Solve the Multiple Longest Common Subsequence Problem.

**Input**: Three DNA strings of length at most 10.

**Output**: The length of a longest common subsequence of these three strings, followed by a multiple alignment of the three strings corresponding to such an alignment.

**Sample Input**:

```
ATATCCG
TCCGA
ATGTACTG
```

**Sample Output**:

```
3
ATATCC-G-
---TCC-GA
ATGTACTG-
```

In [2]:
def MultipleLCS(v, w, u):
    n = len(v)
    m = len(w)
    l = len(u)
    
    s = [[[0 for _ in range(l + 1)] for _ in range(m + 1)] for _ in range(n + 1)]
    backtrack = [[[ -1 for _ in range(l + 1)] for _ in range(m + 1)] for _ in range(n + 1)]
    
    for i in range(n + 1):
        for j in range(m + 1):
            for k in range(l + 1):
                if i == 0 and j == 0 and k == 0:
                    continue
                
                candidates = []
                
                # 0: (i-1, j, k)
                if i > 0: candidates.append((s[i-1][j][k], 0))
                
                # 1: (i, j-1, k)
                if j > 0: candidates.append((s[i][j-1][k], 1))
                
                # 2: (i, j, k-1)
                if k > 0: candidates.append((s[i][j][k-1], 2))
                
                # 3: (i-1, j-1, k)
                if i > 0 and j > 0: candidates.append((s[i-1][j-1][k], 3))
                
                # 4: (i-1, j, k-1)
                if i > 0 and k > 0: candidates.append((s[i-1][j][k-1], 4))
                
                # 5: (i, j-1, k-1)
                if j > 0 and k > 0: candidates.append((s[i][j-1][k-1], 5))
                
                # 6: (i-1, j-1, k-1)
                if i > 0 and j > 0 and k > 0:
                    match = 1 if v[i-1] == w[j-1] == u[k-1] else 0
                    candidates.append((s[i-1][j-1][k-1] + match, 6))
                
                if not candidates:
                    continue
                    
                # Sort by score (desc), then by move type (desc) to prioritize 6 > ... > 0
                candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
                
                best_score, best_move = candidates[0]
                s[i][j][k] = best_score
                backtrack[i][j][k] = best_move
                
    # Backtrack
    align_v = ""
    align_w = ""
    align_u = ""
    
    i, j, k = n, m, l
    while i > 0 or j > 0 or k > 0:
        move = backtrack[i][j][k]
        
        if move == 6:
            align_v += v[i-1]
            align_w += w[j-1]
            align_u += u[k-1]
            i -= 1; j -= 1; k -= 1
        elif move == 5:
            align_v += '-'
            align_w += w[j-1]
            align_u += u[k-1]
            j -= 1; k -= 1
        elif move == 4:
            align_v += v[i-1]
            align_w += '-'
            align_u += u[k-1]
            i -= 1; k -= 1
        elif move == 3:
            align_v += v[i-1]
            align_w += w[j-1]
            align_u += '-'
            i -= 1; j -= 1
        elif move == 2:
            align_v += '-'
            align_w += '-'
            align_u += u[k-1]
            k -= 1
        elif move == 1:
            align_v += '-'
            align_w += w[j-1]
            align_u += '-'
            j -= 1
        elif move == 0:
            align_v += v[i-1]
            align_w += '-'
            align_u += '-'
            i -= 1
            
    return s[n][m][l], align_v[::-1], align_w[::-1], align_u[::-1]

In [56]:
# Sample Input
v = "ATATCCG"
w = "TCCGA"
u = "ATGTACTG"

score, align_v, align_w, align_u = MultipleLCS(v, w, u)

print(score)
print(align_v)
print(align_w)
print(align_u)

# Test Assertion
expected_score = 3
assert score == expected_score, f"Expected score {expected_score}, but got {score}"
print("Test passed!")

3
AT-ATCCG-
-T---CCGA
ATGTACTG-
Test passed!


In [59]:
# Test Dataset
test_dataset_filename = 'dataset_30203_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    v = lines[0].strip()
    w = lines[1].strip()
    u = lines[2].strip()
    
    score, align_v, align_w, align_u = MultipleLCS(v, w, u)
    
    print(score)
    print(align_v)
    print(align_w)
    print(align_u)
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

4
-CAGCCG-----CTA
CGATCAG-----C-C
--A-C-GTAATAC-G


In [3]:
#Coursera Quiz Questions

v = "CCAATACGAC"
w = "GCCTTACGCT"
u = "CCCTAGCGGC"

score, align_v, align_w, align_u = MultipleLCS(v, w, u)

print(score)
print(align_v)
print(align_w)
print(align_u)

7
-CCAATA-CGAC-
GCC-TTA-CG-CT
-CC-CTAGCGGC-
